# Lab 2: Parallelization and Evaluator-Optimizer Pattern

## Features
1. *Functionise repetitive code blocks*
2. *Parallelism*
3. Orchestration based on model capability
4. 

In [ ]:
import os
import json
import asyncio
from typing import Dict, List, Tuple, Any
from dotenv import load_dotenv
from openai import OpenAI, AsyncOpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [12]:
load_dotenv(override=True)

True

In [13]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")


OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI


In [55]:
# =============================
# Worker Preparation
# =============================

def evaluator_prepare_configs():
    """
    Returns a list of model configurations ready for parallel execution.
    """

    configs = []

    configs.append({
        'client': AsyncOpenAI(),
        'model_name': 'gpt-5-nano',
        'provider': 'openai',
        'call_type': 'chat.completions'
    })

    configs.append({
        'client': AsyncOpenAI(),
        'model_name': 'gpt-5.4-mini',
        'provider': 'openai',
        'call_type': 'chat.completions'
    })

    configs.append({
        'client': AsyncOpenAI(),
        'model_name': 'gpt-5.5',
        'provider': 'openai',
        'call_type': 'chat.completions'
    })



    return configs

# Prepare global config config dictionaries
MODEL_CONFIGS = evaluator_prepare_configs()
MODEL_CONFIGS_BY_NAME = {cfg["model_name"]: cfg for cfg in MODEL_CONFIGS}


In [56]:
# Call a single model and get a response
async def get_response(config: Dict[str, Any], messages: List[Dict]) -> Tuple[str, str]:
    """
    Gets a response from a single model.
    """
    if config['call_type']=='chat.completions':
        response = await config['client'].chat.completions.create(
            model=config['model_name'],
            messages=messages
        )
        return response.choices[0].message.content

In [57]:
# Given a question prompt, cycle through the models and async gather responses 
async def get_responses_for_question(question: str) -> Dict[str, str]:
    """
    Gets responses from all models in parallel for a given question.
    """

    messages = [
        {"role": "system", "content": "You are a helpful assistant for answering questions."},
        {"role": "user", "content": question}
    ]

    tasks = []
    for config in MODEL_CONFIGS:
        tasks.append(get_response(config, messages))
    
    responses = await asyncio.gather(*tasks)

    return {config['model_name']: response for config, response in zip(MODEL_CONFIGS, responses)}

In [58]:
def get_question() -> str:
    """
    Generates a question prompt for the models.
    """
    prompt = "Generate a question that tests the model's ability to understand and reason about the complex scenarios. The question should require multi-step reasoning and not be answerable with a simple fact lookup. The question should be clear and concise, and should not contain any ambiguous language."

    messages = [
        {"role": "system", "content": "You are a helpful assistant for generating questions."},
        {"role": "user", "content": prompt} 
    ]

    model_config = MODEL_CONFIGS[0]  # Just use the first model config for the question generation

    if model_config['call_type']=='chat.completions':
        response = OpenAI().chat.completions.create(
            model=model_config['model_name'],
            messages=messages
        )
        return response.choices[0].message.content.strip()



In [59]:
def rank_responses(res) -> List[Tuple[str, str]]:
    """
    Use an LLM the rank the responses from best to worst.
    """

    prompt = "Rank the following responses to the question from best to worst. Consider factors such as correctness, completeness, reasoning ability, and clarity. Respond only with the ranked list of model names, without any additional commentary.\n\n"
    for model_name, response in res.items():
        prompt += f"Model: {model_name}\nResponse: {response}\n\n"
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant for evaluating and ranking model responses"},
        {"role": "user", "content": prompt}
    ]

    model_config = MODEL_CONFIGS[0]

    if model_config['call_type']=='chat.completions':
        response = OpenAI().chat.completions.create(
            model=model_config['model_name'],
            messages=messages
        )
        ranked_models = response.choices[0].message.content.strip().split("\n")
        ranked_models = [model.strip() for model in ranked_models if model.strip() in res]
        return [(model, res[model]) for model in ranked_models]

In [60]:
question = get_question()

print(f"Question has been generated for the models: {question}")

responses = await get_responses_for_question(question)

print("Responses have been generated for the question.")

ranked_responses = rank_responses(responses)

print("Responses have been ranked. Here are the results:\n")
for i, (model, response) in enumerate(ranked_responses, start=1):
    print(f"{i}. {model}")


Question has been generated for the models: Four people A, B, C, D sit in a row of four seats labeled 1 through 4 from left to right. The following constraints apply:
- A cannot sit in seat 1 or 4.
- D sits immediately to the left of A.
- B sits somewhere to the left of C.
- C sits adjacent to D if and only if A sits in an even-numbered seat.

Question: How many valid seating arrangements satisfy all constraints? Provide the total count, and give one explicit arrangement (specify the seat number for each of A, B, C, D) for every valid case.


/tmp/ipykernel_161624/4129203425.py:5: RuntimeWarning: coroutine 'get_responses_for_question' was never awaited
  responses = await get_responses_for_question(question)


Responses have been generated for the question.
Responses have been ranked. Here are the results:

1. gpt-5.5
2. gpt-5.5
3. gpt-5.4-mini
4. gpt-5-nano
